In [65]:
import pandas as pd
import numpy as np
import networkx as nx
import pickle
import ast
import os

from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [66]:
# Load cleaned jobs
jobs = pd.read_csv("../data/processed/jobs_cleaned.csv")

# Load new synthetic interactions (4,000 users)
interactions = pd.read_csv(
    "../data/processed/user_interactions_v2.csv"
)

# Load content-based similarity matrix
combined_similarity = np.load(
    "../models/combined_similarity.npy"
)

# Load updated graph model
with open("../models/job_graph.pkl", "rb") as file:
    job_graph = pickle.load(file)

print("All datasets and models loaded successfully!")

print("Jobs:", jobs.shape)
print("Interactions:", interactions.shape)
print("Unique users:", interactions['user_id'].nunique())
print("Content similarity matrix:", combined_similarity.shape)
print("Graph nodes:", job_graph.number_of_nodes())
print("Graph edges:", job_graph.number_of_edges())

All datasets and models loaded successfully!
Jobs: (1020, 13)
Interactions: (80224, 6)
Unique users: 4000
Content similarity matrix: (1020, 1020)
Graph nodes: 5020
Graph edges: 80224


*Rebuild the collaborative filtering matrices*

In [67]:
# Create user-job interaction matrix
user_job_matrix = interactions.pivot_table(
    index='user_id',
    columns='job_id',
    values='rating',
    fill_value=0
)

# Calculate user-user similarity
user_similarity = cosine_similarity(user_job_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_job_matrix.index,
    columns=user_job_matrix.index
)

print("Collaborative filtering matrices created!")
print("User-job matrix:", user_job_matrix.shape)
print("User similarity matrix:", user_similarity_df.shape)

Collaborative filtering matrices created!
User-job matrix: (4000, 1020)
User similarity matrix: (4000, 4000)


*Prepare job ID mapping*

In [68]:
# Map job IDs to their corresponding matrix positions
job_id_to_index = {
    job_id: index
    for index, job_id in enumerate(jobs['id'])
}

# Map matrix positions back to job IDs
index_to_job_id = {
    index: job_id
    for index, job_id in enumerate(jobs['id'])
}

print("Job ID mappings created successfully!")

Job ID mappings created successfully!


**Build the hybrid recommendation function**

In [69]:
def normalize_score_dict(scores):

    if not scores:
        return {}

    values = np.array(list(scores.values()), dtype=float)

    min_score = values.min()
    max_score = values.max()

    # Equal scores provide no ranking information
    if max_score == min_score:
        return {job_id: 0.0 for job_id in scores}

    normalized = (values - min_score) / (max_score - min_score)

    return dict(zip(scores.keys(), normalized))


def recommend_hybrid(
    user_id,
    jobs,
    interactions,
    user_job_matrix,
    user_similarity_df,
    combined_similarity,
    job_graph,
    job_id_to_index,
    top_n=5,
    top_k=10,
    content_weight=0.50,
    collaborative_weight=0.30,
    graph_weight=0.20
):

    # Check whether user exists
    if user_id not in user_job_matrix.index:
        return "User not found!"

    # Check that weights add up to 1
    total_weight = (
        content_weight +
        collaborative_weight +
        graph_weight
    )

    if not np.isclose(total_weight, 1.0):
        raise ValueError("Recommendation weights must add up to 1.")

    # Identify jobs already interacted with
    user_ratings = user_job_matrix.loc[user_id]

    already_interacted = set(
        user_ratings[user_ratings > 0].index
    )

    # ----------------------------------
    # 1. CONTENT-BASED SCORES
    # ----------------------------------

    user_history = user_ratings[user_ratings > 0]

    content_scores = {}

    for job_id in job_id_to_index:

        if job_id in already_interacted:
            continue

        candidate_index = job_id_to_index[job_id]

        historical_scores = []
        historical_weights = []

        for previous_job_id, rating in user_history.items():

            previous_index = job_id_to_index[previous_job_id]

            similarity = combined_similarity[
                candidate_index,
                previous_index
            ]

            historical_scores.append(similarity)
            historical_weights.append(rating)

        if historical_weights:
            score = np.average(
                historical_scores,
                weights=historical_weights
            )

            content_scores[job_id] = score

    # ----------------------------------
    # 2. COLLABORATIVE FILTERING SCORES
    # ----------------------------------

    similar_users = user_similarity_df[user_id].drop(user_id)

    similar_users = (
        similar_users[similar_users > 0]
        .sort_values(ascending=False)
        .head(top_k)
    )

    collaborative_scores = {}

    if not similar_users.empty:

        neighbor_ratings = user_job_matrix.loc[
            similar_users.index
        ]

        for job_id in user_job_matrix.columns:

            if job_id in already_interacted:
                continue

            ratings = neighbor_ratings[job_id]

            active_neighbors = ratings > 0

            if not active_neighbors.any():
                continue

            relevant_ratings = ratings[active_neighbors]

            relevant_similarities = similar_users[
                active_neighbors
            ]

            score = np.dot(
                relevant_ratings,
                relevant_similarities
            ) / np.sum(np.abs(relevant_similarities))

            collaborative_scores[job_id] = score

    # ----------------------------------
    # 3. GRAPH-BASED SCORES
    # ----------------------------------

    graph_scores = {}

    if user_id in job_graph:

        personalization = {
            node: 0 for node in job_graph.nodes()
        }

        personalization[user_id] = 1

        pagerank_scores = nx.pagerank(
            job_graph,
            alpha=0.85,
            personalization=personalization,
            weight='weight'
        )

        for node, score in pagerank_scores.items():

            if job_graph.nodes[node]['node_type'] != 'job':
                continue

            job_id = int(node.replace('JOB_', ''))

            if job_id not in already_interacted:
                graph_scores[job_id] = score

    # ----------------------------------
    # 4. NORMALIZE THE THREE SCORES
    # ----------------------------------

    content_normalized = normalize_score_dict(content_scores)

    collaborative_normalized = normalize_score_dict(
        collaborative_scores
    )

    graph_normalized = normalize_score_dict(graph_scores)

    # ----------------------------------
    # 5. COMBINE THE SCORES
    # ----------------------------------

    all_candidate_ids = (
        set(content_scores) |
        set(collaborative_scores) |
        set(graph_scores)
    )

    hybrid_results = []

    for job_id in all_candidate_ids:

        content_score = content_normalized.get(job_id, 0)

        collaborative_score = (
            collaborative_normalized.get(job_id, 0)
        )

        graph_score = graph_normalized.get(job_id, 0)

        final_score = (
            content_weight * content_score +
            collaborative_weight * collaborative_score +
            graph_weight * graph_score
        )

        hybrid_results.append({
            'job_id': job_id,
            'content_score': content_score,
            'collaborative_score': collaborative_score,
            'graph_score': graph_score,
            'hybrid_score': final_score
        })

    # ----------------------------------
    # 6. RANK AND RETURN RECOMMENDATIONS
    # ----------------------------------

    recommendations = pd.DataFrame(hybrid_results)

    recommendations = recommendations.sort_values(
        by='hybrid_score',
        ascending=False
    ).head(top_n)

    recommendations = recommendations.merge(
        jobs[
            ['id', 'title', 'company', 'location', 'employment_type']
        ],
        left_on='job_id',
        right_on='id',
        how='left'
    )

    return recommendations[
        [
            'job_id',
            'title',
            'company',
            'location',
            'employment_type',
            'content_score',
            'collaborative_score',
            'graph_score',
            'hybrid_score'
        ]
    ]

*Generate hybrid recommendations*

In [70]:
test_user = 'USER_001'

hybrid_recommendations = recommend_hybrid(
    user_id=test_user,
    jobs=jobs,
    interactions=interactions,
    user_job_matrix=user_job_matrix,
    user_similarity_df=user_similarity_df,
    combined_similarity=combined_similarity,
    job_graph=job_graph,
    job_id_to_index=job_id_to_index,
    top_n=5,
    top_k=10,
    content_weight=0.50,
    collaborative_weight=0.30,
    graph_weight=0.20
)

hybrid_recommendations

,job_id,title,company,location,employment_type,content_score,collaborative_score,graph_score,hybrid_score
0,346,Software Engineer Full Stack & Application Development I (Full Time) – United States,Cisco,"Dallas, TX",Full-time,1.000000,1.000000,1.000000,1.000000
1,335,Software Engineer,Winaxis LLC,"Lewes, DE",Full-time,0.781005,0.784330,0.606038,0.747009
2,282,"Software Engineer, Ground Software",Muon Space,"San Jose, CA",Full-time,0.883588,0.491634,0.684137,0.726112
3,436,Software Engineer Intern,Leidos,"Reston, VA",Full-time,0.898609,0.483042,0.590792,0.712376
4,143,2026 Dark Wolf Summer Internship,Dark Wolf,"Herndon, VA",Internship,0.650596,0.677992,0.856058,0.699907


In [71]:
hybrid_recommendations['calculated_score'] = (
    0.50 * hybrid_recommendations['content_score'] +
    0.30 * hybrid_recommendations['collaborative_score'] +
    0.20 * hybrid_recommendations['graph_score']
)

print(
    hybrid_recommendations[
        ['job_id', 'hybrid_score', 'calculated_score']
    ]
)

   job_id  hybrid_score  calculated_score
0     346      1.000000          1.000000
1     335      0.747009          0.747009
2     282      0.726112          0.726112
3     436      0.712376          0.712376
4     143      0.699907          0.699907


In [72]:
test_user = 'USER_097'

hybrid_recommendations = recommend_hybrid(
    user_id=test_user,
    jobs=jobs,
    interactions=interactions,
    user_job_matrix=user_job_matrix,
    user_similarity_df=user_similarity_df,
    combined_similarity=combined_similarity,
    job_graph=job_graph,
    job_id_to_index=job_id_to_index,
    top_n=5,
    top_k=10,
    content_weight=0.50,
    collaborative_weight=0.30,
    graph_weight=0.20
)

hybrid_recommendations

,job_id,title,company,location,employment_type,content_score,collaborative_score,graph_score,hybrid_score
0,444,Data Engineer,Traackr,United States,Full-time,1.000000,0.169762,0.348780,0.620684
1,155,Generative AI Solution Consultant,The Hartford,"Charlotte, NC",Full-time,0.564879,1.000000,0.012863,0.585012
2,63,Machine Learning Engineer,RADAR,San Diego Metropolitan Area,Full-time,0.684597,0.240541,0.795862,0.573633
3,1,Machine Learning Engineering Internship - Summer 2026,Viant Technology,"Irvine, CA",Internship,0.731462,0.000000,0.948910,0.555513
4,311,Data Engineer Intern - Tampa (Hybrid) - $20./hr,Confidential,Greater Tampa Bay Area,Temporary,0.635935,0.500000,0.394994,0.546966


In [73]:
if isinstance(hybrid_recommendations, pd.DataFrame):

    previously_interacted = set(
        user_job_matrix.loc[test_user][
            user_job_matrix.loc[test_user] > 0
        ].index
    )

    recommended_job_ids = set(
        hybrid_recommendations['job_id']
    )

    print(
        "Previously interacted jobs recommended:",
        len(previously_interacted.intersection(recommended_job_ids))
    )

    print(
        "Number of recommendations:",
        len(hybrid_recommendations)
    )

Previously interacted jobs recommended: 0
Number of recommendations: 5


In [74]:
os.makedirs("../data/processed", exist_ok=True)

hybrid_recommendations.to_csv(
    "../data/processed/hybrid_recommendations.csv",
    index=False
)

print("Hybrid recommendations saved successfully!")

Hybrid recommendations saved successfully!
